# PhilHealth Cauayan: Sentiment Model Training
Combined FiReCS + SentiTaglish (human-annotated Taglish reviews). Compares Naive Bayes, SVM, KNN and Random Forest.

In [ ]:
# CELL 1: Install packages (run once, then restart the kernel)
# scikit-learn must match the version in the Django app's requirements.txt (1.9.0),
# otherwise the saved .pkl may fail to load in the system.
%pip install scikit-learn==1.9.0 pandas matplotlib nltk joblib

In [ ]:
# CELL 2: Imports and settings
import re, json, time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)

RANDOM_STATE = 42                      # fixed seed so results are reproducible
LABEL_NAMES = {0: "Negative", 1: "Neutral", 2: "Positive"}
print("scikit-learn version:", sklearn.__version__)   # should print 1.9.0

In [ ]:
# CELL 3: Load and combine the two datasets
# FiReCS:       0=Negative, 1=Neutral, 2=Positive (split into train/test files)
# SentiTaglish: 1=Negative, 2=Neutral, 3=Positive, 4=Mixed
# Mixed -> Neutral follows FiReCS's own convention (labels agree 100% on the shared reviews).
firecs = pd.concat([pd.read_csv("FiReCS_train_set.csv"),
                    pd.read_csv("FiReCS_test_set.csv")], ignore_index=True)
firecs = firecs.rename(columns={"review": "sentence"})
firecs["source"] = "FiReCS"

senti = pd.read_csv("SentiTaglish_ProductsAndServices.csv")
senti = senti.rename(columns={"review": "sentence", "sentiment": "label"})
senti["label"] = senti["label"].map({1: 0, 2: 1, 3: 2, 4: 1})
senti["source"] = "SentiTaglish"

df = pd.concat([firecs, senti], ignore_index=True)
print("FiReCS:", len(firecs), "| SentiTaglish:", len(senti), "| Combined:", len(df))

In [ ]:
# CELL 4: Data cleaning
# Remove empty rows, invalid labels and duplicates (the two datasets share ~1,890 reviews).
before = len(df)
df = df.dropna(subset=["sentence", "label"])
df["sentence"] = df["sentence"].astype(str).str.strip()
df = df[df["sentence"] != ""]
df["label"] = df["label"].astype(int)
df = df[df["label"].isin([0, 1, 2])]
df = df[~df["sentence"].str.lower().duplicated()]      # case-insensitive duplicate removal
df = df.reset_index(drop=True)
print(f"Removed {before - len(df)} rows -> {len(df)} rows remain")   # expect 19,112
df.to_csv("combined_dataset.csv", index=False)                         # keep a copy for the paper's appendix

In [ ]:
# CELL 5: Explore the data (Data Understanding phase of CRISP-DM)
counts = df["label"].map(LABEL_NAMES).value_counts()
print(counts, "\n")
print(df.groupby("source")["label"].value_counts().unstack().rename(columns=LABEL_NAMES), "\n")

df["n_words"] = df["sentence"].str.split().str.len()
print("Words per review:\n", df["n_words"].describe().round(1))

counts.plot(kind="bar", color=["#d9534f", "#999999", "#5cb85c"], title="Class distribution")
plt.ylabel("Reviews"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150); plt.show()

In [ ]:
# CELL 6: Stop words (English + Tagalog)
# Negation and intensity words are KEPT because they change the sentiment:
# "hindi maayos" (not okay) must not become "maayos" (okay).
tagalog_sw = {
    "ang","ng","nang","sa","mga","na","at","ay","si","ni","kay","sina","nina","kina",
    "ako","ko","akin","ikaw","ka","mo","iyo","siya","sya","niya","nya","kanya",
    "kami","tayo","namin","natin","amin","atin","kayo","ninyo","nyo","inyo",
    "sila","nila","kanila","ito","iyan","yan","iyon","yon","yun","yung","dito","diyan","dyan","doon","don",
    "po","ho","opo","naman","lang","lamang","din","rin","pa","ba","nga","daw","raw",
    "kasi","para","pag","kapag","kung","o","eh","ah","e","yata","pala",
    "ano","sino","saan","kailan","paano","may","mayroon","meron","nasa",
}
KEEP = {"no","not","nor","never","nothing","none","cannot","but","very","too","against",
        "hindi","di","wala","walang","huwag","wag","ayaw","pero","sobra","sobrang"}
STOP_WORDS = (set(ENGLISH_STOP_WORDS) | tagalog_sw) - KEEP
print(len(STOP_WORDS), "stop words")

In [ ]:
# CELL 7: Text preprocessing (normalization, tokenization, stop-word removal, stemming)
# IMPORTANT: the Django app must run this exact same function on new comments.
stemmer = SnowballStemmer("english")

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()                              # normalization: lowercase
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)       # "sobraaaa" -> "sobraa"
    text = re.sub(r"[^a-z\s]", " ", text)            # remove numbers, punctuation, emojis
    tokens = text.split()                             # tokenization
    tokens = [t for t in tokens if t not in STOP_WORDS]   # stop-word removal
    return " ".join(stemmer.stem(t) for t in tokens)      # stemming (English words; Tagalog mostly unchanged)

# Before/after examples for the paper
for s in df["sentence"].sample(5, random_state=7):
    print("RAW  :", s[:110]); print("CLEAN:", preprocess(s)[:110]); print()

In [ ]:
# CELL 8: Apply preprocessing to every review
t = time.time()
df["clean"] = df["sentence"].apply(preprocess)
df = df[df["clean"].str.len() > 0].reset_index(drop=True)   # drop reviews that became empty
print(f"{len(df)} rows ready ({time.time() - t:.1f}s)")

In [ ]:
# CELL 9: Split into training (80%) and testing (20%) sets
# stratify keeps the same Negative/Neutral/Positive ratio in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.20, stratify=df["label"], random_state=RANDOM_STATE)
print("Training:", len(X_train), "| Testing:", len(X_test))
print(y_test.map(LABEL_NAMES).value_counts())

In [ ]:
# CELL 10: Define the models
# Each model is a Pipeline: TF-IDF (turns text into numbers) -> classifier.
# ngram_range=(1,2) also uses word pairs, e.g. "hindi maayos", "ang tagal".
def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)

models = {
    "Naive Bayes":   Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB(alpha=0.3))]),
    "SVM (Linear)":  Pipeline([("tfidf", tfidf()), ("clf", LinearSVC(C=0.5, random_state=RANDOM_STATE))]),
    "KNN":           Pipeline([("tfidf", tfidf()), ("clf", KNeighborsClassifier(n_neighbors=15, metric="cosine", weights="distance"))]),
    "Random Forest": Pipeline([("tfidf", tfidf()), ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE))]),
}

In [ ]:
# CELL 11: Train every model and compare (Random Forest takes the longest)
rows = []
for name, pipe in models.items():
    t = time.time()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="macro"),
        "Recall":    recall_score(y_test, pred, average="macro"),
        "F1-Score":  f1_score(y_test, pred, average="macro"),
        "Time (s)":  round(time.time() - t, 1),
    })
    print(f"done: {name}")

results = pd.DataFrame(rows).sort_values("F1-Score", ascending=False).reset_index(drop=True)
results.to_csv("model_comparison.csv", index=False)     # table for Chapter 4
results.round(4)

In [ ]:
# CELL 12: Per-class precision / recall / F1 for each model
for name, pipe in models.items():
    print("=" * 15, name, "=" * 15)
    print(classification_report(y_test, pipe.predict(X_test),
                                target_names=list(LABEL_NAMES.values()), digits=4))

In [ ]:
# CELL 13: Confusion matrices (Objective 2 of the study)
fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
for ax, (name, pipe) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pipe.predict(X_test), display_labels=list(LABEL_NAMES.values()),
        cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

In [ ]:
# CELL 14: Choose the final model + 5-fold cross-validation
# Default = highest F1 from Cell 11. To force a model, write e.g. BEST = "Random Forest".
BEST = results.loc[0, "Model"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(models[BEST], df["clean"], df["label"], cv=cv, scoring="f1_macro", n_jobs=-1)
print(f"{BEST}: 5-fold macro F1 = {scores.mean():.4f} ± {scores.std():.4f}")
print("Per fold:", scores.round(4))

In [ ]:
# CELL 15: Error analysis: what does the model get wrong?
best = models[BEST]
test_df = df.loc[X_test.index, ["sentence", "label"]].copy()
test_df["predicted"] = best.predict(X_test)
wrong = test_df[test_df["label"] != test_df["predicted"]]
print(f"{len(wrong)} of {len(test_df)} test reviews misclassified\n")
for _, r in wrong.sample(10, random_state=1).iterrows():
    print(f"[actual {LABEL_NAMES[r.label]:8s} | predicted {LABEL_NAMES[r.predicted]:8s}] {r.sentence[:110]}")

In [ ]:
# CELL 16: Try it on PhilHealth-style comments (add your own real comments here)
samples = [
    "Ang bilis ng proseso, mabait si ate sa counter. Salamat po!",
    "Sobrang tagal ng pila, 3 oras ako naghintay tapos walang paliwanag.",
    "Nag-update lang po ako ng MDR.",
    "The staff was very helpful and accommodating.",
    "Hindi maayos ang serbisyo, bastos pa yung guard.",
    "Okay naman, pero sana dagdagan ang upuan sa waiting area.",
]
for s in samples:
    print(f"{LABEL_NAMES[int(best.predict([preprocess(s)])[0])]:8s} | {s}")

In [ ]:
# CELL 17: Retrain the chosen model on ALL data and save it for the Django app
final_model = models[BEST]
final_model.fit(df["clean"], df["label"])

joblib.dump(final_model, "sentiment_model.pkl")                 # the model
with open("stopwords.txt", "w", encoding="utf-8") as f:          # the app must use the same stop words
    f.write("\n".join(sorted(STOP_WORDS)))
with open("model_info.json", "w") as f:                          # record for the paper
    json.dump({
        "model": BEST,
        "datasets": ["FiReCS (ccosme/FiReCS)", "SentiTaglish (ccosme/SentiTaglishProductsAndServices)"],
        "rows": int(len(df)),
        "test_metrics": results.loc[0].drop("Model").to_dict(),
        "cv_macro_f1_mean": float(scores.mean()),
        "labels": LABEL_NAMES,
        "sklearn_version": sklearn.__version__,
    }, f, indent=2)
print("Saved: sentiment_model.pkl, stopwords.txt, model_info.json")

In [ ]:
# CELL 18: Sanity check: reload the saved file the same way the app will
loaded = joblib.load("sentiment_model.pkl")
print(loaded)
print(LABEL_NAMES[int(loaded.predict([preprocess("Ang bait ng staff, mabilis ang transaction")])[0])])